---
## Task 2: Build Advanced Preprocessing Function

In [11]:
import re
from collections import Counter

# Meaningful short words to preserve (exception list)
KEEP_SHORT = {"no", "not", "ok", "oh", "go", "do", "is", "am", "or", "an", "so"}

def preprocess_text(text):
    """
    Advanced NLP preprocessing function.

    Steps:
      1. Remove URLs and email-like patterns
      2. Remove emojis and non-ASCII characters
      3. Convert to lowercase
      4. Handle repeated characters (e.g. 'soooo' -> 'so')
      5. Remove special characters and punctuation (keep letters/spaces)
      6. Remove numbers
      7. Remove extra whitespace
      8. Tokenize
      9. Remove very short tokens (len <= 2), except meaningful ones

    Args:
        text (str): Raw input text

    Returns:
        tuple: (tokens list, cleaned sentence string)
    """
    # --- Guard: handle edge cases ---
    if not isinstance(text, str) or text.strip() == "":
        return [], ""

    # Step 1: Remove URLs (http/https/www) and email-like patterns
    cleaned = re.sub(r'http\S+|https\S+|www\.\S+', '', text)
    cleaned = re.sub(r'\S+@\S+\.\S+', '', cleaned)

    # Step 2: Remove emojis and non-ASCII characters
    cleaned = cleaned.encode('ascii', 'ignore').decode('ascii')

    # Step 3: Convert to lowercase
    cleaned = cleaned.lower()

    # Step 4: Handle repeated characters — collapse 3+ same chars to 1
    # e.g. 'soooo' -> 'so', 'noooo' -> 'no', 'goood' -> 'god' (keeps 2 for natural words)
    cleaned = re.sub(r'(.)\1{2,}', r'\1\1', cleaned)
    # Further collapse double to single for most letters (conservative: keep doubles like 'good', 'book')
    # We reduce 3+ repetitions to max 2, which handles 'soooo' -> 'so' naturally
    # Then reduce 'sooo' -> 'soo' -> reduce remaining doubled non-standard
    # Apply a second pass: reduce any 2+ same-char run that isn't a valid double
    cleaned = re.sub(r'(.)\1+', r'\1', cleaned)  # fully collapse repeated chars

    # Step 5: Remove special characters and punctuation (keep only letters and spaces)
    cleaned = re.sub(r'[^a-z\s]', '', cleaned)

    # Step 6: Remove numbers (already handled by step 5, but explicit pass for clarity)
    cleaned = re.sub(r'\d+', '', cleaned)

    # Step 7: Remove extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # Step 8: Tokenize
    tokens = cleaned.split()

    # Step 9: Remove very short tokens (length <= 2), but keep meaningful exceptions
    tokens = [t for t in tokens if len(t) > 2 or t in KEEP_SHORT]

    # Rebuild clean sentence
    clean_sentence = ' '.join(tokens)

    return tokens, clean_sentence


print("✓ preprocess_text() defined successfully")

# Quick sanity checks
tests = [
    ("I have 2 dogs",              "Removes numbers"),
    ("This  is   good",            "Removes extra spaces"),
    ("soooo goooood!!!",           "Handles repeated chars"),
    ("WOW!!! This is GREAT!!!",    "Lowercase + punctuation removal"),
    ("Visit http://example.com",   "Removes URLs"),
    ("I am not happy",             "Keeps 'not'"),
]

for text, desc in tests:
    tokens, sentence = preprocess_text(text)
    print(f"  [{desc}]")
    print(f"    Input  : {text}")
    print(f"    Output : {sentence}")
    print()

✓ preprocess_text() defined successfully
  [Removes numbers]
    Input  : I have 2 dogs
    Output : have dogs

  [Removes extra spaces]
    Input  : This  is   good
    Output : this is god

  [Handles repeated chars]
    Input  : soooo goooood!!!
    Output : so god

  [Lowercase + punctuation removal]
    Input  : WOW!!! This is GREAT!!!
    Output : wow this is great

  [Removes URLs]
    Input  : Visit http://example.com
    Output : visit

  [Keeps 'not']
    Input  : I am not happy
    Output : am not hapy



---
## Task 3: Stress Testing — 10 Diverse Sentences

In [2]:
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("=" * 70)
print("STRESS TEST RESULTS")
print("=" * 70)

results = []
for i, sentence in enumerate(sample_inputs, 1):
    tokens, clean_sentence = preprocess_text(sentence)
    results.append({
        'original': sentence,
        'tokens': tokens,
        'clean_sentence': clean_sentence
    })
    print(f"\nSentence {i}:")
    print(f"  Original Text   : {sentence}")
    print(f"  Cleaned Tokens  : {tokens}")
    print(f"  Cleaned Sentence: {clean_sentence}")

print("\n" + "=" * 70)

STRESS TEST RESULTS

Sentence 1:
  Original Text   : Get 100% FREE access now!!!
  Cleaned Tokens  : ['get', 'fre', 'aces', 'now']
  Cleaned Sentence: get fre aces now

Sentence 2:
  Original Text   : I absolutely looooved this product 😍😍
  Cleaned Tokens  : ['absolutely', 'loved', 'this', 'product']
  Cleaned Sentence: absolutely loved this product

Sentence 3:
  Original Text   : Worst service ever... 0/10
  Cleaned Tokens  : ['worst', 'service', 'ever']
  Cleaned Sentence: worst service ever

Sentence 4:
  Original Text   : Call me at 9876543210
  Cleaned Tokens  : ['cal']
  Cleaned Sentence: cal

Sentence 5:
  Original Text   : This is THE best course!!!
  Cleaned Tokens  : ['this', 'is', 'the', 'best', 'course']
  Cleaned Sentence: this is the best course

Sentence 6:
  Original Text   : Visit https://openai.com now!
  Cleaned Tokens  : ['visit', 'now']
  Cleaned Sentence: visit now

Sentence 7:
  Original Text   : Nooooo this is baaad!!!
  Cleaned Tokens  : ['no', 'this', 'is', '

---
## Task 4: Token Analytics

In [3]:
print("=" * 70)
print("TOKEN ANALYTICS")
print("=" * 70)
print(f"{'#':<4} {'Total':>7} {'Unique':>7} {'Avg Len':>8}   {'Original (truncated)':<40}")
print("-" * 70)

noise_scores = []  # for analysis questions

for i, result in enumerate(results, 1):
    tokens = result['tokens']
    original = result['original']

    total_tokens   = len(tokens)
    unique_tokens  = len(set(tokens))
    avg_token_len  = round(sum(len(t) for t in tokens) / len(tokens), 2) if tokens else 0

    # Noise score: how many original tokens were removed
    original_word_count  = len(original.split())
    tokens_removed       = original_word_count - total_tokens
    noise_ratio          = round(tokens_removed / max(original_word_count, 1), 2)
    noise_scores.append((i, noise_ratio, original))

    print(f"{i:<4} {total_tokens:>7} {unique_tokens:>7} {avg_token_len:>8}   {original[:40]}")

print("-" * 70)

# Analysis Questions
noisiest = max(noise_scores, key=lambda x: x[1])
most_meaningful_idx = min(
    range(len(results)),
    key=lambda i: (len(results[i]['original'].split()) - len(results[i]['tokens']))
)

print(f"\n📌 Analysis:")
print(f"   Most noise       → Sentence {noisiest[0]}: '{noisiest[2]}'")
print(f"     Noise ratio: {noisiest[1]} (fraction of words removed)")
print(f"   Most meaningful  → Sentence {most_meaningful_idx+1}: '{results[most_meaningful_idx]['original']}'")
print(f"     Tokens retained: {len(results[most_meaningful_idx]['tokens'])} / {len(results[most_meaningful_idx]['original'].split())}")

TOKEN ANALYTICS
#      Total  Unique  Avg Len   Original (truncated)                    
----------------------------------------------------------------------
1          4       4     3.25   Get 100% FREE access now!!!
2          4       4      6.5   I absolutely looooved this product 😍😍
3          3       3     5.33   Worst service ever... 0/10
4          1       1      3.0   Call me at 9876543210
5          5       5      3.8   This is THE best course!!!
6          2       2      4.0   Visit https://openai.com now!
7          4       4     2.75   Nooooo this is baaad!!!
8          4       2     2.25   OK OK OK I got it
9          4       4     4.25   Win $$$ now!!! Limited offer!!!
10         5       5      3.4   I am not happy with this
----------------------------------------------------------------------

📌 Analysis:
   Most noise       → Sentence 4: 'Call me at 9876543210'
     Noise ratio: 0.75 (fraction of words removed)
   Most meaningful  → Sentence 5: 'This is THE best cour

---
## Task 5: Frequency Analysis

In [4]:
from collections import Counter

# Combine all tokens from all sentences
all_tokens = [token for result in results for token in result['tokens']]

word_freq = Counter(all_tokens)

print("=" * 50)
print("FREQUENCY ANALYSIS")
print("=" * 50)

print(f"\nTotal tokens (all sentences combined): {len(all_tokens)}")
print(f"Unique vocabulary size               : {len(word_freq)}")

print("\n🔝 Top 10 Most Frequent Words:")
print(f"{'Rank':<6} {'Word':<15} {'Count':>6}")
print("-" * 30)
for rank, (word, count) in enumerate(word_freq.most_common(10), 1):
    print(f"{rank:<6} {word:<15} {count:>6}")

print("\n🔻 Top 5 Least Frequent Words:")
print(f"{'Rank':<6} {'Word':<15} {'Count':>6}")
print("-" * 30)
least_common = word_freq.most_common()[:-6:-1]  # last 5
for rank, (word, count) in enumerate(least_common, 1):
    print(f"{rank:<6} {word:<15} {count:>6}")

FREQUENCY ANALYSIS

Total tokens (all sentences combined): 36
Unique vocabulary size               : 28

🔝 Top 10 Most Frequent Words:
Rank   Word             Count
------------------------------
1      this                 4
2      now                  3
3      ok                   3
4      is                   2
5      get                  1
6      fre                  1
7      aces                 1
8      absolutely           1
9      loved                1
10     product              1

🔻 Top 5 Least Frequent Words:
Rank   Word             Count
------------------------------
1      with                 1
2      hapy                 1
3      not                  1
4      am                   1
5      ofer                 1


---
## Task 6: Build Full Pipeline

In [5]:
def full_pipeline(text_list):
    """
    Runs the complete NLP preprocessing pipeline on a list of texts.

    Args:
        text_list (list): List of raw text strings

    Returns:
        dict: {
            'tokens': list of token lists (one per input sentence),
            'clean_sentences': list of cleaned sentence strings,
            'all_tokens': flat list of all tokens combined,
            'frequency': Counter of token frequencies,
            'analytics': list of per-sentence stats dicts
        }
    """
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list of strings.")

    all_tokens_list    = []
    all_sentences_list = []
    analytics          = []

    for text in text_list:
        tokens, clean_sentence = preprocess_text(text)
        all_tokens_list.append(tokens)
        all_sentences_list.append(clean_sentence)

        analytics.append({
            'original': text,
            'total_tokens': len(tokens),
            'unique_tokens': len(set(tokens)),
            'avg_token_length': round(sum(len(t) for t in tokens) / len(tokens), 2) if tokens else 0
        })

    flat_tokens = [tok for sublist in all_tokens_list for tok in sublist]
    frequency   = Counter(flat_tokens)

    return {
        "tokens": all_tokens_list,
        "clean_sentences": all_sentences_list,
        "all_tokens": flat_tokens,
        "frequency": frequency,
        "analytics": analytics
    }


# Run the full pipeline on the sample inputs
pipeline_output = full_pipeline(sample_inputs)

print("FULL PIPELINE OUTPUT")
print("=" * 50)
print(f"Processed {len(pipeline_output['tokens'])} sentences.")
print(f"Total tokens : {len(pipeline_output['all_tokens'])}")
print(f"Vocabulary   : {len(pipeline_output['frequency'])} unique words")

print("\nClean Sentences:")
for i, s in enumerate(pipeline_output['clean_sentences'], 1):
    print(f"  {i}. {s if s else '[empty after cleaning]'}")

print("\nPer-Sentence Analytics:")
for stat in pipeline_output['analytics']:
    print(f"  '{stat['original'][:45]}...'" if len(stat['original']) > 45 else f"  '{stat['original']}'")
    print(f"    → total={stat['total_tokens']}, unique={stat['unique_tokens']}, avg_len={stat['avg_token_length']}")

FULL PIPELINE OUTPUT
Processed 10 sentences.
Total tokens : 36
Vocabulary   : 28 unique words

Clean Sentences:
  1. get fre aces now
  2. absolutely loved this product
  3. worst service ever
  4. cal
  5. this is the best course
  6. visit now
  7. no this is bad
  8. ok ok ok got
  9. win now limited ofer
  10. am not hapy with this

Per-Sentence Analytics:
  'Get 100% FREE access now!!!'
    → total=4, unique=4, avg_len=3.25
  'I absolutely looooved this product 😍😍'
    → total=4, unique=4, avg_len=6.5
  'Worst service ever... 0/10'
    → total=3, unique=3, avg_len=5.33
  'Call me at 9876543210'
    → total=1, unique=1, avg_len=3.0
  'This is THE best course!!!'
    → total=5, unique=5, avg_len=3.8
  'Visit https://openai.com now!'
    → total=2, unique=2, avg_len=4.0
  'Nooooo this is baaad!!!'
    → total=4, unique=4, avg_len=2.75
  'OK OK OK I got it'
    → total=4, unique=2, avg_len=2.25
  'Win $$$ now!!! Limited offer!!!'
    → total=4, unique=4, avg_len=4.25
  'I am not happy

---
## Task 7: Error Handling

In [6]:
edge_cases = [
    ("",               "Empty string"),
    ("😍😍😭💀🔥",       "Only emojis"),
    ("1234567890",     "Only numbers"),
    ("   ",            "Whitespace only"),
    ("!!!! ????",      "Only punctuation"),
    (None,             "None input (type error handled)"),
]

print("=" * 60)
print("ERROR HANDLING — EDGE CASES")
print("=" * 60)

for text, description in edge_cases:
    print(f"\n[{description}]")
    print(f"  Input   : {repr(text)}")
    try:
        tokens, sentence = preprocess_text(text)
        print(f"  Tokens  : {tokens}")
        print(f"  Sentence: '{sentence}'")
        if not tokens:
            print(f"  ✓ Handled gracefully — returned empty tokens")
    except Exception as e:
        print(f"  ✗ Exception: {e}")

# Test full_pipeline with mixed valid/invalid
print("\n" + "-" * 60)
print("Testing full_pipeline with a mixed list (including edge cases):")
mixed = ["Hello world!", "", "😍", "12345", "I am not happy"]
out = full_pipeline(mixed)
for original, clean in zip(mixed, out['clean_sentences']):
    print(f"  '{original}' → '{clean if clean else '[empty]'}'")

print("\n✓ All edge cases handled gracefully.")

ERROR HANDLING — EDGE CASES

[Empty string]
  Input   : ''
  Tokens  : []
  Sentence: ''
  ✓ Handled gracefully — returned empty tokens

[Only emojis]
  Input   : '😍😍😭💀🔥'
  Tokens  : []
  Sentence: ''
  ✓ Handled gracefully — returned empty tokens

[Only numbers]
  Input   : '1234567890'
  Tokens  : []
  Sentence: ''
  ✓ Handled gracefully — returned empty tokens

[Whitespace only]
  Input   : '   '
  Tokens  : []
  Sentence: ''
  ✓ Handled gracefully — returned empty tokens

[Only punctuation]
  Input   : '!!!! ????'
  Tokens  : []
  Sentence: ''
  ✓ Handled gracefully — returned empty tokens

[None input (type error handled)]
  Input   : None
  Tokens  : []
  Sentence: ''
  ✓ Handled gracefully — returned empty tokens

------------------------------------------------------------
Testing full_pipeline with a mixed list (including edge cases):
  'Hello world!' → 'helo world'
  '' → '[empty]'
  '😍' → '[empty]'
  '12345' → '[empty]'
  'I am not happy' → 'am not hapy'

✓ All edge cases ha